In [ ]:
def clean_tickets(tickets):
    """Drop Duplicates, empty and personal columns, tickets without a message (includes both na and "" values), and make each category one spelling"""
    cleaned = tickets.drop_duplicates().copy()
    cleaned = cleaned.drop(columns=["Customer", "Email", "Phone", "Internal Notes"])
    cleaned["Message"] = cleaned["Message"].str.strip()
    cleaned = cleaned[cleaned["Message"] != ""].copy()
    cleaned = cleaned.dropna(subset=["Message"]).copy()
    cleaned["Category"] = cleaned["Category"].str.lower().str.strip()
    return cleaned

In [ ]:
import sqlite3
import pandas as pd

class TicketStore:
    """Store cleaned tickets and AI predictions in SQLite."""

    def __init__(self, filename):
        self._connection = sqlite3.connect(filename)
        self._cursor = self._connection.cursor()

        self._cursor.execute("""
            CREATE TABLE IF NOT EXISTS triage (
                ticket_number TEXT PRIMARY KEY,
                message TEXT,
                true_category TEXT,
                ai_category TEXT,
                ai_category_again TEXT,
                ai_reply TEXT,
                rating INTEGER)""")

        self._connection.commit()

    def load(self, tickets):
        """Save cleaned tickets and prepare an empty triage table."""
        tickets.to_sql("tickets",self._connection,if_exists="replace",index=False)
        self._cursor.execute("DELETE FROM triage")
        self._connection.commit()

    def sample(self, n):
        """Return the first n tickets as tuples."""
        self._cursor.execute("""
            SELECT "Ticket Number", "Message", "Category" FROM tickets LIMIT ?""", (n,))

        return self._cursor.fetchall()

    def save_triage(self, ticket_number, message, true_category,ai_category, ai_category_again, ai_reply, rating=None):
        """Save an AI prediction and reply."""
        self._cursor.execute("""
            INSERT OR REPLACE INTO triage (ticket_number, message, true_category,
             ai_category, ai_category_again, ai_reply, rating) VALUES (?, ?, ?, ?, ?, ?, ?)""", (ticket_number, message, true_category, ai_category, ai_category_again, ai_reply, rating))

        self._connection.commit()

    def results(self):
        """Return the triage table as a DataFrame."""

        return pd.read_sql_query(
            "SELECT * FROM triage",
            self._connection)

    def close(self):
        """Let go of the database file."""
        self._connection.close()


    def save_rating(self, ticket_number, rating):
      """Save the rating of a Gemini reply."""
      self._cursor.execute("""
        UPDATE triage
        SET rating = ?
        WHERE ticket_number = ?
    """, (rating, ticket_number))

      self._connection.commit()


In [ ]:
def clean_text(text):
    """Returns the text in lower case with no surrounding spaces (beginning and end)."""
    return str(text).lower().strip()


KEYWORDS = {
    "return": "return request",
    "delivery": "delivery problem",
    "invoice": "payment issue",
    "password": "account help"
}


def stand_in_classify(message):
    """Classify a message using simple keywords."""
    message = clean_text(message)

    for keyword in KEYWORDS:
        if keyword in message:
            return KEYWORDS[keyword]

    return "product question"


class SupportDesk:
    """Classify support tickets using simple keywords."""

    def __init__(self, store):
        self.store = store

    def classify(self, message):
        """Classify a ticket using the stand-in model."""
        return stand_in_classify(message)

    def draft_reply(self, message):
        """Simply reply to the request."""
        return "Thank you for your request. Our support team will review it and get back to you as soon as possible."

    def triage(self, tickets, save = False):
        """Classify the tickets and return the results."""
        answers = []

        for number, message, true_category in tickets:
            category = self.classify(message)
            reply = self.draft_reply(message)

            answers.append((number, true_category, category, reply))

            if save:
              self.store.save_triage(
                  number, message, true_category, category, None, reply)

        return answers

In [ ]:
import time

class GeminiDesk(SupportDesk):
    """SupportDesk using Gemini instead of keywords."""

    def __init__(self, store, client, model, config):
        super().__init__(store)
        self.client = client
        self.model = model
        self.config = config

    def ask_gemini(self, prompt):
        """Send a prompt to Gemini and return the answer"""
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
            config=self.config)
        time.sleep(5)
        return response.text

    def classify(self, message):
        """Classify the ticket using Gemini."""
        prompt = (
          "Read the following customer message and classify it into one of these categories: "
          "return request, delivery problem, product question, payment issue, account help. "
          "Only return the category value. "
          f"Message: {message}"
      )

        return self.ask_gemini(prompt).lower().strip()

    def draft_reply(self, message):
        """Create a customer support reply using Gemini."""
        prompt = (
            "Write a short and simple customer support reply to this message. "
            f"Message: {message}"
        )

        return self.ask_gemini(prompt).strip()